# nginx_who_mortality_ingest


In [ ]:
from __future__ import annotations

import os
import re
import sys
import subprocess
from dataclasses import dataclass
from io import StringIO

try:
    import requests
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests"], check=True)
    import requests

import pandas as pd
from pyspark.sql import DataFrame, functions as F


BASE_URL_ENV = "WHO_MORTALITY_BASE_URL"
DEFAULT_BASE_URL = "https://q7k0jp9j-8000.use2.devtunnels.ms/"
TARGET_SCHEMA = "sandbox"

INDICATOR_HEADER_PREFIX = (
    "Indicator Code,Indicator Name,Year,Sex,Age group code,Age Group,Number,"
    "Percentage of cause-specific deaths out of total deaths"
)

POPULATION_HEADER_PREFIX = "Year,Age group code,Age group,Sex,Population"


@dataclass(frozen=True)
class SourceSpec:
    file_name: str
    remote_path: str
    header_prefix: str
    target_table: str
    table_kind: str
    source_kind: str


SOURCE_SPECS = (
    SourceSpec(
        "deaths_by_age_group_gtm.csv",
        "deaths_by_age_group_gtm.csv",
        INDICATOR_HEADER_PREFIX,
        "sandbox.raw_who_mortality_deaths_by_age_group_gtm",
        "deaths_by_age_group",
        "indicator",
    ),
    SourceSpec(
        "population_distribution_gtm.csv",
        "population_distribution_gtm.csv",
        POPULATION_HEADER_PREFIX,
        "sandbox.raw_who_mortality_population_distribution_gtm",
        "population_distribution",
        "population",
    ),
)


def snake_case(value: str) -> str:
    text = re.sub(r"[^0-9A-Za-z]+", "_", value.strip())
    text = re.sub(r"_+", "_", text)
    return text.strip("_").lower()


def normalize_text(value: str | None) -> str | None:
    if value is None:
        return None

    text = str(value).replace("\ufeff", "").strip().strip('"')
    text = re.sub(r"\s+", " ", text)

    return text or None


def source_url(base_url: str, remote_path: str) -> str:
    return f"{base_url.rstrip('/')}/{remote_path.lstrip('/')}"


def fetch_text(url: str) -> str:
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    return response.text


def split_metadata_and_csv(raw_text: str, header_prefix: str) -> tuple[dict[str, str], str]:
    lines = [
        line.rstrip("\r")
        for line in raw_text.splitlines()
        if normalize_text(line)
    ]

    header_idx = next(
        i
        for i, line in enumerate(lines)
        if normalize_text(line).lower().startswith(header_prefix.lower())
    )

    metadata: dict[str, str] = {}

    for line in lines[:header_idx]:
        if ":" not in line:
            continue

        key, value = line.split(":", 1)
        key_norm = snake_case(key)
        value_norm = normalize_text(value)

        if (
            key_norm
            in {
                "region_code",
                "region_name",
                "country_code",
                "country_name",
                "export_date",
                "source_location",
            }
            and value_norm
        ):
            metadata[key_norm] = value_norm

    cleaned_csv = "\n".join(line.rstrip(",") for line in lines[header_idx:])

    return metadata, cleaned_csv


def read_csv_from_text(csv_text: str) -> DataFrame:
    # Databricks Connect: no se puede escribir a DBFS root (deshabilitado) ni
    # usar spark.read sobre el FS del cluster. Se parsea el CSV con pandas en el
    # cliente y se envía al cluster vía createDataFrame. inferSchema equivalente
    # al default de pandas (int/float/str según el contenido).
    pdf = pd.read_csv(StringIO(csv_text), keep_default_na=False, encoding="utf-8")
    return spark.createDataFrame(pdf)


def add_raw_metadata(
    df: DataFrame,
    metadata: dict[str, str],
    spec: SourceSpec,
    base_url: str,
) -> DataFrame:
    return (
        df
        .withColumn("_country_code", F.lit(metadata.get("country_code")))
        .withColumn("_country_name", F.lit(metadata.get("country_name")))
        .withColumn("_region_code", F.lit(metadata.get("region_code")))
        .withColumn("_region_name", F.lit(metadata.get("region_name")))
        .withColumn("_source_export_date", F.lit(metadata.get("export_date")))
        .withColumn("_source_location", F.lit(metadata.get("source_location")))
        .withColumn("_source_file", F.lit(spec.file_name))
        .withColumn("_source_url", F.lit(source_url(base_url, spec.remote_path)))
        .withColumn("_table_kind", F.lit(spec.table_kind))
        .withColumn("_source_kind", F.lit(spec.source_kind))
        .withColumn("_ingestion_timestamp", F.current_timestamp())
    )


def write_delta_table(df: DataFrame, table_name: str) -> None:
    # Las columnas se guardan con sus nombres originales del CSV (con espacios y
    # guiones, ej. "Indicator Code"). Delta rechaza esos caracteres salvo que se
    # habilite Column Mapping. La normalización a snake_case ocurre en staging.
    # Se recrea la tabla para poder fijar el modo de mapeo aunque exista una
    # versión previa sin él.
    spark.sql(f"DROP TABLE IF EXISTS {table_name}")
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option("delta.columnMapping.mode", "name")
        .option("delta.minReaderVersion", "2")
        .option("delta.minWriterVersion", "5")
        .saveAsTable(table_name)
    )


def get_base_url() -> str:
    return os.getenv(BASE_URL_ENV, DEFAULT_BASE_URL).rstrip("/")


def ensure_target_schema() -> None:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}")


def process_source(spec: SourceSpec, base_url: str) -> DataFrame:
    raw_text = fetch_text(source_url(base_url, spec.remote_path))
    metadata, cleaned_csv = split_metadata_and_csv(raw_text, spec.header_prefix)

    # Columnas se guardan con sus nombres originales del CSV; la normalización
    # a snake_case ocurre en staging (notebooks/staging/who_mortality.py).
    df = read_csv_from_text(cleaned_csv)

    return add_raw_metadata(df, metadata, spec, base_url)


def main() -> None:
    ensure_target_schema()

    base_url = get_base_url()

    for spec in SOURCE_SPECS:
        df = process_source(spec, base_url)
        display(df)
        write_delta_table(df, spec.target_table)


if __name__ == "__main__":
    main()